# P7 / 01: capitalisation ranks, rank-space residuals and rolling network fits

Protocol: [`001_claude_p7_rank_space_preregistration.md`](../../discussions/Rank_Space/001_claude_p7_rank_space_preregistration.md).

- **Capitalisation:** split-adjusted (not dividend-adjusted) price × point-in-time SEC shares outstanding, on today's split basis.
- **Ranks:** 1–400 each close.
- **Rank returns:** the change in the k-th largest cap. These are not tradable.
- **Residuals:** PCA with 1 factor.
- **Network:** P6's CNN + Transformer.
- **Two arms:** `RS-paper` (gross mean-variance on rank returns) and `RS-net` (Sharpe of the tradable rank-mapped book net of measured costs).
- **Fits:** rolling 750 / 250 / 125, 3 seeds. Every fit is cached.

In [1]:
import json, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, p1, p6, p7, s04, s05b
from equity_trading.alpaca import ALPACA_VERSION, download_years, session_schedule
from equity_trading.data import vendor_dir, work_dir

SYNTHETIC = os.environ.get("P7_SYNTHETIC") == "1"
# a kernel started before `git pull` keeps the old modules in memory: refuse to run on them
REQUIRED = {"p7": "0.1.1", "p6": "0.1.0"}
_have = {"p7": p7.P7_VERSION, "p6": p6.P6_VERSION}
assert _have == REQUIRED, f"stale equity_trading modules {_have}, expected {REQUIRED}: restart the kernel (Kernel > Restart) and run again"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
CFG = p7.P7Config()
ARMS = {"RS-paper": "paper", "RS-net": "net"}
CACHE = work_dir() / "results" / ("p7_synthetic" if SYNTHETIC else "p7") / f"v{p7.P7_VERSION}"
CACHE.mkdir(parents=True, exist_ok=True)
if SYNTHETIC:
    CFG = replace(CFG, n_ranks=80, train_days=300, valid_days=100, first_test="2017-01-03", end="2017-12-29",
                  seeds=(0,), epochs=3, patience=3, n_placebo=50, n_boot=2000)
import torch
from equity_trading import p7_model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else " -- CUDA not available"))
print(f"synthetic={SYNTHETIC} | ranks {CFG.n_ranks} | arms {list(ARMS)} | seeds {CFG.seeds}")

torch 2.14.0+cu126 | device cuda (NVIDIA GeForce GTX 1070)
synthetic=False | ranks 400 | arms ['RS-paper', 'RS-net'] | seeds (0, 1, 2)


## 1. Returns, split-only prices, shares outstanding, capitalisation, costs

In [2]:
t0 = time.time()
if SYNTHETIC:
    R, CAP = p7.synthetic_market(n_days=1800, n_stocks=120, start="2011-01-03")
    COST = pd.DataFrame(2e-4, index=R.index, columns=R.columns)
else:
    constituents = pd.read_csv(ROOT / "src" / "equity_trading" / "reference" / "sp500_constituents_20260913.csv")
    STOCKS = sorted(constituents["Symbol"])
    sessions = pd.DatetimeIndex(pd.to_datetime(list(session_schedule(list(range(2011, 2026))).index)))
    # total returns (as P6): Yahoo adjusted before 2016-01-04, Alpaca adjusted after
    YH = p6.load_yahoo_daily(STOCKS, "2005-01-01", "2016-01-06", verbose=False)
    download_years(STOCKS, "1Day", list(range(2016, 2026)), workers=8)
    AC, _, _ = p1.load_daily(STOCKS, list(range(2016, 2026)))
    R = p6.splice_returns(YH, AC, sessions, CFG.splice)
    # split-adjusted, not dividend-adjusted prices (for capitalisation)
    YS = p7.yahoo_split_prices(STOCKS, CFG.data_start, "2016-01-06")
    AS = p7.alpaca_split_daily(STOCKS, list(range(2016, 2026)))
    cut = pd.Timestamp(CFG.splice)
    PX = AS.reindex(index=sessions, columns=R.columns)
    ys = YS.reindex(index=sessions, columns=R.columns)
    PX.loc[sessions < cut] = ys.loc[sessions < cut]
    # shares outstanding (SEC, point in time) on today's split basis
    REC = p7.sec_shares(STOCKS)
    SPL, CUR = p7.yahoo_splits_and_shares(STOCKS)
    SH, DIAG = p7.shares_daily(REC, SPL, CUR, sessions, CFG.unit_band)
    DIAG.to_csv(CACHE / "shares_diagnostics.csv", index=False)
    CAP = (PX * SH.reindex(index=sessions, columns=R.columns))
    R = R.loc[:, CAP.notna().any()]
    CAP = CAP[R.columns]
    qc = vendor_dir("Alpaca") / "quote_samples" / "15Min_w3s"
    sess = s05b.sample_sessions([2017, 2018, 2019], 4, seed=42) + s05b.sample_sessions(list(range(2020, 2026)), 4, seed=42)
    samples = pd.concat([pd.read_parquet(qc / f"{d}.parquet") for d in sess], ignore_index=True)
    COST = p1.daily_cost(R.index, list(R.columns), s05b.cost_model(samples))
    print(f"shares: {len(DIAG)} symbols with SEC counts; filings dropped by the C1 cleaning: {int(DIAG['dropped'].sum()) if len(DIAG) else 0}; "
          f"rescaled by the Yahoo unit check: {DIAG.loc[DIAG.rescaled, 'symbol'].tolist()}")
    late = DIAG[pd.to_datetime(DIAG["first_filed"]) > "2012-06-30"].sort_values("first_filed")
    print(f"{len(late)} symbols whose first share count is after 2012-06 (no cap before):", late["symbol"].tolist()[:40])
valid_caps = CAP.notna().sum(axis=1)
print(f"{R.shape[1]} stocks, {len(R)} sessions; stocks with a cap per year (median):")
display(valid_caps.groupby(valid_caps.index.year).median().to_frame("stocks with cap").T)
print(f"days with >= {CFG.n_ranks} caps: {(valid_caps >= CFG.n_ranks).mean():.1%}; ready in {time.time() - t0:.0f}s")

1Day: 0 batches to download (0 symbol-years), 8 parallel requests
Yahoo split-only prices: 503/503 cached; downloading 0
  Alpaca split-adjusted 1Day 2016: 0 symbols downloaded
  Alpaca split-adjusted 1Day 2017: 0 symbols downloaded
  Alpaca split-adjusted 1Day 2018: 0 symbols downloaded
  Alpaca split-adjusted 1Day 2019: 0 symbols downloaded
  Alpaca split-adjusted 1Day 2020: 0 symbols downloaded
  Alpaca split-adjusted 1Day 2021: 0 symbols downloaded
  Alpaca split-adjusted 1Day 2022: 0 symbols downloaded
  Alpaca split-adjusted 1Day 2023: 0 symbols downloaded
  Alpaca split-adjusted 1Day 2024: 0 symbols downloaded
  Alpaca split-adjusted 1Day 2025: 0 symbols downloaded
  SEC shares 50/503, 64s
  SEC shares 100/503, 144s
  SEC shares 150/503, 212s
  SEC shares 200/503, 270s
  SEC shares 250/503, 330s
  SEC shares 300/503, 389s
  SEC shares 350/503, 451s
  SEC shares 400/503, 515s
  SEC shares 450/503, 584s
  SEC shares 500/503, 650s
  SEC shares 503/503, 654s
  no CIK for 2: ['PSKY',

,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
stocks with cap,373.0,401.0,410.0,419.0,425.0,435.0,439.0,444.0,456.0,463.0,472.0,477.0,480.0,486.0,490.0


days with >= 400 caps: 91.2%; ready in 808s


## 2. Rank-space dataset (cached)

In [3]:
t0 = time.time()
ds_path = CACHE / f"dataset_p7{p7.P7_VERSION}_K{CFG.n_ranks}.npz"
SCHED = p6.fit_schedule(R.index, CFG)
need_from = max(0, SCHED[0]["train"][0] - CFG.lookback)
if ds_path.exists():
    DS = p7.load_dataset(ds_path)
else:
    DS = p7.build_rank_dataset(R, CAP, COST, CFG, start_row=need_from)
    p7.save_dataset(DS, ds_path)
rows = np.arange(need_from, len(R) - 2)
rr, tr = DS["ret"][rows], DS["tret"][rows]
m = np.isfinite(rr) & np.isfinite(tr)
print(f"dataset {DS['x'].shape}; usable rank-days {DS['ok'][need_from:].mean():.1%}; {time.time() - t0:.0f}s")
print(f"rank return vs tradable return of the same rank: corr {np.corrcoef(rr[m], tr[m])[0, 1]:.3f}, "
      f"s.d. {np.std(rr[m]):.4f} vs {np.std(tr[m]):.4f}")
print(f"{len(SCHED)} fits per arm; first predicts from {SCHED[0]['first_day']}")
# C1 data-quality gate (protocol 002): rank returns must look like returns before any training
ratio_sd = np.std(rr[m]) / np.std(tr[m])
big = np.abs(rr[m]) > 0.25
print(f"rank-return s.d. / tradable s.d. = {ratio_sd:.2f}; share of rank-days with |rank return| > 25%: {big.mean():.4%}")
if not SYNTHETIC:
    worst = np.argsort(-np.nanmax(np.abs(np.where(np.isfinite(DS['rank_return']), DS['rank_return'], 0)), 1))[:10]
    print("days with the largest |rank return|:", [(str(DS["index"][i].date()), round(float(np.nanmax(np.abs(DS["rank_return"][i]))), 3)) for i in worst])
assert ratio_sd < 3, "rank returns are not clean (C1 gate): stop and report; do not train"

  PCA residuals 500/3294 days, 9s
  PCA residuals 1000/3294 days, 20s
  PCA residuals 1500/3294 days, 34s
  PCA residuals 2000/3294 days, 45s
  PCA residuals 2500/3294 days, 55s
  PCA residuals 3000/3294 days, 67s
dataset (3772, 400, 30); usable rank-days 95.8%; 75s
rank return vs tradable return of the same rank: corr 0.615, s.d. 0.0139 vs 0.0195
19 fits per arm; first predicts from 2017-01-03
rank-return s.d. / tradable s.d. = 0.71; share of rank-days with |rank return| > 25%: 0.0004%
days with the largest |rank return|: [('2019-02-27', 0.615), ('2016-01-04', 0.388), ('2016-04-12', 0.331), ('2022-02-03', 0.258), ('2023-05-25', 0.244), ('2016-02-08', 0.238), ('2025-04-09', 0.22), ('2024-02-02', 0.202), ('2016-05-06', 0.196), ('2016-02-24', 0.196)]


## 3. Rolling fits, both arms

In [4]:
INFO, t_all, n_done, n_tot = [], time.time(), 0, len(ARMS) * len(SCHED)
for arm, mode in ARMS.items():
    SC = np.full(DS["ok"].shape, np.nan, np.float32)
    for j, f in enumerate(SCHED):
        tr_, va_, pr_ = (np.arange(*f[k]) for k in ("train", "valid", "predict"))
        tr_, va_ = tr_[DS["ok"][tr_].sum(1) > 0], va_[DS["ok"][va_].sum(1) > 0]
        seed_scores = []
        for s in CFG.seeds:
            p = CACHE / f"{arm}_fit{j:02d}_seed{s}.npz"
            if p.exists():
                z = np.load(p, allow_pickle=True); seed_scores.append(z["scores"]); info = json.loads(str(z["info"]))
            else:
                t1 = time.time()
                model, info = p7_model.fit(DS, tr_, va_, seed=s, mode=mode, gamma=CFG.gamma, epochs=CFG.epochs,
                                           patience=CFG.patience, lr=CFG.lr, chunk=CFG.chunk_days, device=DEVICE)
                sc = p7_model.predict_scores(model, DS, pr_, chunk=CFG.chunk_days)
                info["seconds"] = time.time() - t1
                np.savez(p, scores=sc, info=json.dumps(info)); seed_scores.append(sc)
            INFO.append({"arm": arm, "fit": j, "seed": s, "first_day": f["first_day"], **info})
        with np.errstate(invalid="ignore"):
            zs = [(x - np.nanmean(x, 1, keepdims=True)) / np.nanstd(x, 1, keepdims=True) for x in seed_scores]
            SC[pr_] = np.nanmean(np.stack(zs), 0)
        n_done += 1
        el = time.time() - t_all
        print(f"{arm} fit {j + 1}/{len(SCHED)} ({f['first_day']}): valid by seed "
              f"{[round(i['best_valid'], 2) for i in INFO[-len(CFG.seeds):]]}; {el / 60:.1f} min, "
              f"~{el / n_done * (n_tot - n_done) / 60:.0f} min left", flush=True)
    np.savez(CACHE / f"scores_{arm}.npz", scores=SC)
INFO = pd.DataFrame(INFO)
OUT = ROOT / "notebooks" / "Rank_Space" / ("01_synthetic" if SYNTHETIC else "01")
OUT.mkdir(parents=True, exist_ok=True)
INFO.to_csv(OUT / "fits.csv", index=False, float_format="%.5g")
if not SYNTHETIC:
    DIAG.to_csv(OUT / "shares_diagnostics.csv", index=False)
display(INFO.groupby(["arm", "fit"])[["best_valid", "train", "epochs"]].mean().unstack(0).round(3))
(OUT / "manifest.json").write_text(json.dumps({"files": sorted(p.name for p in OUT.iterdir()), "cache": str(CACHE), "device": DEVICE,
                                               "torch": torch.__version__, "p7": p7.P7_VERSION}, indent=2))

RS-paper fit 1/19 (2017-01-03): valid by seed [0.3, 0.28, 0.32]; 5.3 min, ~195 min left
RS-paper fit 2/19 (2017-07-03): valid by seed [0.21, 0.21, 0.2]; 7.7 min, ~139 min left
RS-paper fit 3/19 (2017-12-29): valid by seed [0.23, 0.23, 0.2]; 11.3 min, ~132 min left
RS-paper fit 4/19 (2018-06-29): valid by seed [0.35, 0.29, 0.29]; 20.5 min, ~174 min left
RS-paper fit 5/19 (2018-12-28): valid by seed [0.31, 0.33, 0.27]; 27.5 min, ~181 min left
RS-paper fit 6/19 (2019-06-28): valid by seed [0.31, 0.3, 0.34]; 35.8 min, ~191 min left
RS-paper fit 7/19 (2019-12-26): valid by seed [0.39, 0.49, 0.43]; 46.4 min, ~206 min left
RS-paper fit 8/19 (2020-06-25): valid by seed [0.44, 0.39, 0.37]; 55.9 min, ~210 min left
RS-paper fit 9/19 (2020-12-22): valid by seed [0.29, 0.3, 0.35]; 60.7 min, ~196 min left
RS-paper fit 10/19 (2021-06-23): valid by seed [0.38, 0.32, 0.41]; 66.8 min, ~187 min left
RS-paper fit 11/19 (2021-12-20): valid by seed [0.33, 0.28, 0.32]; 72.1 min, ~177 min left
RS-paper fit 12

C:\Users\Iván\AppData\Local\Temp\ipykernel_2864\1229785172.py:21: RuntimeWarning: Mean of empty slice
  zs = [(x - np.nanmean(x, 1, keepdims=True)) / np.nanstd(x, 1, keepdims=True) for x in seed_scores]
D:\Anaconda\envs\equity-trading\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\Iván\AppData\Local\Temp\ipykernel_2864\1229785172.py:22: RuntimeWarning: Mean of empty slice
  SC[pr_] = np.nanmean(np.stack(zs), 0)


RS-net fit 1/19 (2017-01-03): valid by seed [-0.31, -2.61, -1.31]; 134.1 min, ~121 min left
RS-net fit 2/19 (2017-07-03): valid by seed [0.04, -1.14, 0.6]; 138.4 min, ~112 min left
RS-net fit 3/19 (2017-12-29): valid by seed [0.17, 0.14, 0.2]; 146.6 min, ~107 min left
RS-net fit 4/19 (2018-06-29): valid by seed [-1.41, 0.25, -1.6]; 150.7 min, ~98 min left
RS-net fit 5/19 (2018-12-28): valid by seed [-1.09, 0.93, -0.97]; 158.9 min, ~93 min left
RS-net fit 6/19 (2019-06-28): valid by seed [1.51, 1.08, 0.45]; 166.7 min, ~87 min left
RS-net fit 7/19 (2019-12-26): valid by seed [0.17, 0.88, 1.19]; 174.0 min, ~80 min left
RS-net fit 8/19 (2020-06-25): valid by seed [-1.14, -0.27, 0.05]; 177.8 min, ~72 min left
RS-net fit 9/19 (2020-12-22): valid by seed [-1.4, -0.75, -0.07]; 180.5 min, ~64 min left
RS-net fit 10/19 (2021-06-23): valid by seed [-1.19, -0.94, -1.24]; 184.6 min, ~57 min left
RS-net fit 11/19 (2021-12-20): valid by seed [-0.14, -0.23, 0.2]; 196.8 min, ~52 min left
RS-net fit 12/

best_valid           train           epochs         
arm     RS-net RS-paper RS-net RS-paper  RS-net RS-paper
fit                                                     
0       -1.408    0.299 -1.559    0.318  20.333   38.000
1       -0.168    0.210 -0.717    0.237  23.333   15.667
2        0.171    0.221  0.420    0.265  37.667   22.667
3       -0.918    0.310 -0.356    0.341  20.667   59.667
4       -0.377    0.303  0.437    0.327  41.333   47.333
5        1.011    0.315  0.877    0.324  48.000   57.667
6        0.744    0.438  1.543    0.380  40.333   74.667
7       -0.453    0.401  0.091    0.339  21.000   66.000
8       -0.741    0.317 -1.709    0.351  12.000   33.000
9       -1.120    0.372 -0.348    0.363  17.667   42.333
10      -0.057    0.310  0.944    0.373  54.000   37.000
11      -0.048    0.427  0.966    0.450  50.667   60.333
12       0.047    0.417  0.597    0.433  56.333   54.000
13      -1.496    0.248 -0.004    0.293  34.667   16.667
14      -1.362    0.264 -0.865    0.379  24.000   35.333
15      -0.477    0.399 -0.248    0.425  28.000   60.333
16      -1.256    0.405 -1.054    0.395  14.667   54.667
17      -0.798    0.384 -0.639    0.409  26.000   52.333
18      -1.430    0.403 -0.542    0.425  22.000   56.333

215